# Demand Forecasting and Inventory Optimization for a Grocery Retail Network

## Objective

The objective of this notebook is to understand the structure, quality, and business relevance of the Corporación Favorita grocery sales dataset before performing demand forecasting and inventory optimization.

This stage focuses on:

- Loading all available datasets
- Understanding the size and structure of each table
- Checking date ranges
- Identifying missing values
- Understanding how the tables relate to each other
- Defining which tables will be used for demand analysis, forecasting, and inventory optimization

#### Importing Libraries

In [1]:
import pandas as pd
import numpy as np

#### Setting my folder path

In [2]:
folder = r"C:\Users\DEBIDO\Desktop\favorita-grocery-sales-forecasting"

#### Loading the Files

In [3]:
train = pd.read_csv(folder + r"\train.csv", nrows=100000, low_memory=False)

test = pd.read_csv(folder + r"\test.csv", nrows=100000, low_memory=False)

items = pd.read_csv(folder + r"\items.csv")
stores = pd.read_csv(folder + r"\stores.csv")
transactions = pd.read_csv(folder + r"\transactions.csv")
oil = pd.read_csv(folder + r"\oil.csv")
holidays = pd.read_csv(folder + r"\holidays_events.csv")

In [5]:
print("Train:", train.shape)
print("Test:", test.shape)
print("Items:", items.shape)
print("Stores:", stores.shape)
print("Transactions:", transactions.shape)
print("Oil:", oil.shape)
print("Holidays:", holidays.shape)

Train: (100000, 6)
Test: (100000, 5)
Items: (4100, 4)
Stores: (54, 5)
Transactions: (83488, 3)
Oil: (1218, 2)
Holidays: (350, 6)


In [50]:
print("Train columns:", train.columns.tolist())
print("Test columns:", test.columns.tolist())
print("Items columns:", items.columns.tolist())
print("Stores columns:", stores.columns.tolist())
print("Transactions columns:", transactions.columns.tolist())
print("Oil columns:", oil.columns.tolist())
print("Holidays columns:", holidays.columns.tolist())

Train columns: ['id', 'date', 'store_nbr', 'item_nbr', 'unit_sales', 'onpromotion']
Test columns: ['id', 'date', 'store_nbr', 'item_nbr', 'onpromotion']
Items columns: ['item_nbr', 'family', 'class', 'perishable']
Stores columns: ['store_nbr', 'city', 'state', 'type', 'cluster']
Transactions columns: ['date', 'store_nbr', 'transactions']
Oil columns: ['date', 'dcoilwtico']
Holidays columns: ['date', 'type', 'locale', 'locale_name', 'description', 'transferred']


##### The key relationship between the tables is based on shared columns. The train table connects to the items table through `item_nbr`, to the stores table through `store_nbr`, and to the transactions, oil, and holidays tables through `date`.

In [6]:
train.head()

,id,date,store_nbr,item_nbr,unit_sales,onpromotion
0,0,2013-01-01,25,103665,7.0,NaN
1,1,2013-01-01,25,105574,1.0,NaN
2,2,2013-01-01,25,105575,2.0,NaN
3,3,2013-01-01,25,108079,1.0,NaN
4,4,2013-01-01,25,108701,1.0,NaN


In [49]:
test.head()

,id,date,store_nbr,item_nbr,onpromotion
0,125497040,2017-08-16,1,96995,False
1,125497041,2017-08-16,1,99197,False
2,125497042,2017-08-16,1,103501,False
3,125497043,2017-08-16,1,103520,False
4,125497044,2017-08-16,1,103665,False


In [8]:
items.head()

,item_nbr,family,class,perishable
0,96995,GROCERY I,1093,0
1,99197,GROCERY I,1067,0
2,103501,CLEANING,3008,0
3,103520,GROCERY I,1028,0
4,103665,BREAD/BAKERY,2712,1


In [9]:
stores.head()

,store_nbr,city,state,type,cluster
0,1,Quito,Pichincha,D,13
1,2,Quito,Pichincha,D,13
2,3,Quito,Pichincha,D,8
3,4,Quito,Pichincha,D,9
4,5,Santo Domingo,Santo Domingo de los Tsachilas,D,4


In [10]:
transactions.head()

,date,store_nbr,transactions
0,2013-01-01,25,770
1,2013-01-02,1,2111
2,2013-01-02,2,2358
3,2013-01-02,3,3487
4,2013-01-02,4,1922


In [11]:
oil.head()

,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14
2,2013-01-03,92.97
3,2013-01-04,93.12
4,2013-01-07,93.20


In [12]:
holidays.head()

,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False


#### Convert date columns

In [14]:
train["date"] = pd.to_datetime(train["date"])
test["date"] = pd.to_datetime(test["date"])
transactions["date"] = pd.to_datetime(transactions["date"])
oil["date"] = pd.to_datetime(oil["date"])
holidays["date"] = pd.to_datetime(holidays["date"])

In [15]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   id           100000 non-null  int64         
 1   date         100000 non-null  datetime64[ns]
 2   store_nbr    100000 non-null  int64         
 3   item_nbr     100000 non-null  int64         
 4   unit_sales   100000 non-null  float64       
 5   onpromotion  0 non-null       float64       
dtypes: datetime64[ns](1), float64(2), int64(3)
memory usage: 4.6 MB


In [51]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   id           100000 non-null  int64         
 1   date         100000 non-null  datetime64[ns]
 2   store_nbr    100000 non-null  int64         
 3   item_nbr     100000 non-null  int64         
 4   onpromotion  100000 non-null  bool          
dtypes: bool(1), datetime64[ns](1), int64(3)
memory usage: 3.1 MB


In [52]:
items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4100 entries, 0 to 4099
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   item_nbr    4100 non-null   int64 
 1   family      4100 non-null   object
 2   class       4100 non-null   int64 
 3   perishable  4100 non-null   int64 
dtypes: int64(3), object(1)
memory usage: 128.3+ KB


In [53]:
stores.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   store_nbr  54 non-null     int64 
 1   city       54 non-null     object
 2   state      54 non-null     object
 3   type       54 non-null     object
 4   cluster    54 non-null     int64 
dtypes: int64(2), object(3)
memory usage: 2.2+ KB


In [54]:
transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 83488 entries, 0 to 83487
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   date          83488 non-null  datetime64[ns]
 1   store_nbr     83488 non-null  int64         
 2   transactions  83488 non-null  int64         
dtypes: datetime64[ns](1), int64(2)
memory usage: 1.9 MB


In [55]:
oil.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1218 entries, 0 to 1217
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        1218 non-null   datetime64[ns]
 1   dcoilwtico  1175 non-null   float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 19.2 KB


In [56]:
holidays.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         350 non-null    datetime64[ns]
 1   type         350 non-null    object        
 2   locale       350 non-null    object        
 3   locale_name  350 non-null    object        
 4   description  350 non-null    object        
 5   transferred  350 non-null    bool          
dtypes: bool(1), datetime64[ns](1), object(4)
memory usage: 14.1+ KB


#### Check date ranges

In [16]:
print("Train date range:", train["date"].min(), "to", train["date"].max())
print("Test date range:", test["date"].min(), "to", test["date"].max())
print("Transactions date range:", transactions["date"].min(), "to", transactions["date"].max())
print("Oil date range:", oil["date"].min(), "to", oil["date"].max())
print("Holidays date range:", holidays["date"].min(), "to", holidays["date"].max())

Train date range: 2013-01-01 00:00:00 to 2013-01-04 00:00:00
Test date range: 2017-08-16 00:00:00 to 2017-08-16 00:00:00
Transactions date range: 2013-01-01 00:00:00 to 2017-08-15 00:00:00
Oil date range: 2013-01-01 00:00:00 to 2017-08-31 00:00:00
Holidays date range: 2012-03-02 00:00:00 to 2017-12-26 00:00:00


###### Because only the first 100,000 rows of the train and test files were loaded for initial exploration, the date range shown here reflects the sample loaded into memory, not the full dataset. The full training dataset covers a much longer historical period and will be filtered properly in the next notebook when creating the master dataset.

#### Checking missing values

In [17]:
train.isnull().sum()

id                  0
date                0
store_nbr           0
item_nbr            0
unit_sales          0
onpromotion    100000
dtype: int64

###### The `onpromotion` column contains missing values in the loaded training sample. According to the dataset description, promotion information is missing for part of the training data. Since promotion status is useful for demand analysis, these missing values will be handled during data cleaning. For this project, missing promotion values will be treated as `False`, meaning no promotion recorded.

In [18]:
items.isnull().sum()

item_nbr      0
family        0
class         0
perishable    0
dtype: int64

In [19]:
stores.isnull().sum()

store_nbr    0
city         0
state        0
type         0
cluster      0
dtype: int64

In [20]:
transactions.isnull().sum()

date            0
store_nbr       0
transactions    0
dtype: int64

In [21]:
oil.isnull().sum()

date           0
dcoilwtico    43
dtype: int64

###### The oil table contains 43 missing values in the `dcoilwtico` column. Since oil prices are time-based, these missing values can be handled using forward fill during data cleaning, where missing values are replaced with the most recent available oil price.

In [22]:
holidays.isnull().sum()

date           0
type           0
locale         0
locale_name    0
description    0
transferred    0
dtype: int64

In [23]:
test.isnull().sum()

id             0
date           0
store_nbr      0
item_nbr       0
onpromotion    0
dtype: int64

#### Understanding the sales table

In [25]:
train.head()

,id,date,store_nbr,item_nbr,unit_sales,onpromotion
0,0,2013-01-01,25,103665,7.0,NaN
1,1,2013-01-01,25,105574,1.0,NaN
2,2,2013-01-01,25,105575,2.0,NaN
3,3,2013-01-01,25,108079,1.0,NaN
4,4,2013-01-01,25,108701,1.0,NaN


In [26]:
train.describe()

,id,date,store_nbr,item_nbr,unit_sales,onpromotion
count,100000.000000,100000,100000.000000,1.000000e+05,100000.000000,0.0
mean,49999.500000,2013-01-02 17:57:19.296000,23.538060,6.139470e+05,10.021255,NaN
min,0.000000,2013-01-01 00:00:00,1.000000,1.035010e+05,-27.000000,NaN
25%,24999.750000,2013-01-02 00:00:00,8.000000,3.550900e+05,2.000000,NaN
50%,49999.500000,2013-01-03 00:00:00,19.000000,5.868230e+05,5.000000,NaN
75%,74999.250000,2013-01-03 00:00:00,39.000000,8.723080e+05,11.000000,NaN
max,99999.000000,2013-01-04 00:00:00,54.000000,1.118683e+06,608.909000,NaN
std,28867.657797,NaN,16.422645,3.033932e+05,17.370912,NaN


In [27]:
print("Number of stores:", train["store_nbr"].nunique())
print("Number of items:", train["item_nbr"].nunique())
print("Total unit sales:", train["unit_sales"].sum())

Number of stores: 46
Number of items: 1592
Total unit sales: 1002125.5120000002


#### Checking for negative sales

In [29]:
negative_sales = train[train["unit_sales"] < 0]

print("Negative sales rows:", negative_sales.shape[0])
negative_sales.head()

Negative sales rows: 7


,id,date,store_nbr,item_nbr,unit_sales,onpromotion
10655,10655,2013-01-02,10,456875,-3.0,NaN
46867,46867,2013-01-03,5,559044,-1.0,NaN
50970,50970,2013-01-03,9,365138,-3.0,NaN
71807,71807,2013-01-03,41,812716,-19.0,NaN
71992,71992,2013-01-03,41,1004551,-27.0,NaN


###### Negative values in `unit_sales` represent product returns, according to the dataset description. These rows will be reviewed during cleaning to decide whether they should be removed, adjusted, or analysed separately.

#### Understanding the product families

In [31]:
items.head()

,item_nbr,family,class,perishable
0,96995,GROCERY I,1093,0
1,99197,GROCERY I,1067,0
2,103501,CLEANING,3008,0
3,103520,GROCERY I,1028,0
4,103665,BREAD/BAKERY,2712,1


In [32]:
items["family"].value_counts()

family
GROCERY I                     1334
BEVERAGES                      613
CLEANING                       446
PRODUCE                        306
DAIRY                          242
PERSONAL CARE                  153
BREAD/BAKERY                   134
HOME CARE                      108
DELI                            91
MEATS                           84
HOME AND KITCHEN I              77
LIQUOR,WINE,BEER                73
FROZEN FOODS                    55
POULTRY                         54
HOME AND KITCHEN II             45
EGGS                            41
CELEBRATION                     31
LAWN AND GARDEN                 26
PREPARED FOODS                  26
LADIESWEAR                      21
LINGERIE                        20
AUTOMOTIVE                      20
BEAUTY                          19
PLAYERS AND ELECTRONICS         17
SCHOOL AND OFFICE SUPPLIES      15
GROCERY II                      14
PET SUPPLIES                    14
SEAFOOD                          8
MAGAZINES    

In [33]:
print("Number of product families:", items["family"].nunique())

Number of product families: 33


#### Understanding the store table

In [34]:
stores.head()

,store_nbr,city,state,type,cluster
0,1,Quito,Pichincha,D,13
1,2,Quito,Pichincha,D,13
2,3,Quito,Pichincha,D,8
3,4,Quito,Pichincha,D,9
4,5,Santo Domingo,Santo Domingo de los Tsachilas,D,4


In [35]:
stores["type"].value_counts()

type
D    18
C    15
A     9
B     8
E     4
Name: count, dtype: int64

In [36]:
stores["city"].value_counts()

city
Quito            18
Guayaquil         8
Cuenca            3
Santo Domingo     3
Latacunga         2
Machala           2
Manta             2
Ambato            2
Cayambe           1
Riobamba          1
Ibarra            1
Salinas           1
Puyo              1
Guaranda          1
Quevedo           1
Babahoyo          1
Daule             1
Playas            1
Loja              1
Libertad          1
Esmeraldas        1
El Carmen         1
Name: count, dtype: int64

In [37]:
stores["cluster"].value_counts().sort_index()

cluster
1     3
2     2
3     7
4     3
5     1
6     6
7     2
8     3
9     2
10    6
11    3
12    1
13    4
14    4
15    5
16    1
17    1
Name: count, dtype: int64

#### Understanding the transactions data

In [38]:
transactions.head()

,date,store_nbr,transactions
0,2013-01-01,25,770
1,2013-01-02,1,2111
2,2013-01-02,2,2358
3,2013-01-02,3,3487
4,2013-01-02,4,1922


In [39]:
transactions.describe()

,date,store_nbr,transactions
count,83488,83488.000000,83488.000000
mean,2015-05-20 16:07:40.866232064,26.939237,1694.602158
min,2013-01-01 00:00:00,1.000000,5.000000
25%,2014-03-27 00:00:00,13.000000,1046.000000
50%,2015-06-08 00:00:00,27.000000,1393.000000
75%,2016-07-14 06:00:00,40.000000,2079.000000
max,2017-08-15 00:00:00,54.000000,8359.000000
std,NaN,15.608204,963.286644


In [40]:
print("Number of stores in transactions:", transactions["store_nbr"].nunique())

Number of stores in transactions: 54


#### Understanding oil prices

In [42]:
oil.head()

,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14
2,2013-01-03,92.97
3,2013-01-04,93.12
4,2013-01-07,93.20


In [43]:
oil.describe()

,date,dcoilwtico
count,1218,1175.000000
mean,2015-05-02 12:00:00,67.714366
min,2013-01-01 00:00:00,26.190000
25%,2014-03-03 06:00:00,46.405000
50%,2015-05-02 12:00:00,53.190000
75%,2016-06-30 18:00:00,95.660000
max,2017-08-31 00:00:00,110.620000
std,NaN,25.630476


In [44]:
print("Missing oil prices:", oil["dcoilwtico"].isnull().sum())

Missing oil prices: 43


#### Understanding holidays

In [45]:
holidays.head()

,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False


In [46]:
holidays["type"].value_counts()

type
Holiday       221
Event          56
Additional     51
Transfer       12
Bridge          5
Work Day        5
Name: count, dtype: int64

In [47]:
holidays["locale"].value_counts()

locale
National    174
Local       152
Regional     24
Name: count, dtype: int64

In [48]:
holidays["transferred"].value_counts()

transferred
False    338
True      12
Name: count, dtype: int64

###### Holidays are useful because grocery demand may change around holidays and events.

#### Initial Data Understanding

The train table is the main dataset because it contains historical unit sales, which will be used as demand.

The items table provides product family information, which will help group products into business-friendly categories.

The stores table provides store location and classification details, which will help compare performance across stores.

The transactions table provides store activity information and may help explain demand differences.

The oil table provides an external economic variable.

The holidays table will help identify holiday-related demand patterns.

The dataset does not contain inventory-on-hand levels, so this project will focus on potential replenishment risk rather than confirmed stockouts.

## Data Dictionary Summary

| Dataset | Main Columns | Role in Project |
|---|---|---|
| train | date, store_nbr, item_nbr, unit_sales, onpromotion | Main historical demand data |
| test | date, store_nbr, item_nbr, onpromotion | Future period without sales values |
| items | item_nbr, family, class, perishable | Product classification and inventory grouping |
| stores | store_nbr, city, state, type, cluster | Store profile and location analysis |
| transactions | date, store_nbr, transactions | Store traffic and activity indicator |
| oil | date, dcoilwtico | External economic factor |
| holidays_events | date, type, locale, transferred | Holiday and event impact analysis |

## Table Relationship Plan

The `train` table is the main table for this project because it contains historical unit sales.

The following joins will be used in the next notebook:

- `train` + `items` using `item_nbr`
- `train` + `stores` using `store_nbr`
- `train` + `transactions` using `date` and `store_nbr`
- `train` + `oil` using `date`
- Holiday information will be added using a holiday flag based on `date`

The final output will be a master dataset for demand analysis, forecasting, inventory optimization, and Power BI dashboard development.